In [24]:
import pandas as pd
import numpy as np

# Step 1: Load the dataset
df = pd.read_csv("trend_report.csv")

In [38]:
# Step 2: Convert 'week' to datetime-compatible format (first day of the week)
if df['week'].dtype == 'O':
    df['week'] = pd.to_datetime(df['week'] + '-1', format='%Y-W%W-%w', errors='coerce')

In [39]:
# Step 3: Rename only useful columns
df.rename(columns={
    'col_6': 'engagement_trend',
    'col_7': 'engagement_change_value',
    'col_9': 'checkout_trend',
    'col_10': 'revenue_signal',
    'col_25': 'impressions_total',
    'col_26': 'conversion_shift',
    'col_29': 'revenue_prediction',
    'col_34': 'customer_value_delta',
    'col_46': 'shipping_delay_avg',
    'col_49': 'subscription_length_months',
    'col_50': 'spend_signal'
}, inplace=True)

In [41]:
# Step 4: Keep only the necessary columns
df = df[[
    'week', 'avg_users', 'sales_growth_rate',
    'engagement_trend', 'engagement_change_value',
    'checkout_trend', 'revenue_signal',
    'impressions_total', 'conversion_shift',
    'revenue_prediction', 'customer_value_delta',
    'shipping_delay_avg', 'subscription_length_months',
    'spend_signal'
]]

In [42]:
# Step 5: Fill missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    elif df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())

In [43]:
# Step 6: Cap outliers using IQR
def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return np.clip(series, Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    df[col] = cap_outliers(df[col])

In [44]:
# Step 7: Round float values to 2 decimals
df[num_cols] = df[num_cols].round(2)

In [45]:
# Step 7: Save final cleaned version
df.to_csv("final_cleaned_trend_report.csv", index=False)